In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import wandb
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.training.io import load_model
from einops import rearrange
from jaxtyping import Array, Float
from matplotlib.animation import ArtistAnimation
from tqdm import tqdm


datadir = Path("../../data")
checkpoint_dir = Path("../../checkpoints/cubicflux_1d")

jax.config.update("jax_default_device", jax.devices("gpu")[0])

In [ ]:
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }

In [ ]:
model_paths = {
    "DPOT": [
        "seed=0/26-04-10-14_49_05",
        "seed=10/26-04-16-18_37_37",
        "seed=20/26-04-17-07_57_49",
    ],
    "HyperFluxFNOLocal": [
        "seed=0/26-04-16-00_31_47",
        "seed=10/26-04-17-07_57_49",
        "seed=20/26-04-17-10_50_08",
    ],
    "DISCO": [
        "seed=0/26-04-10-21_26_22",
        "seed=10/26-04-16-18_37_50",
        "seed=20/26-04-16-18_37_50",
    ],
}

In [ ]:
model_paths = {
    "NaiveFluxNO": [
        "seed=0/26-07-26-19_51_22",
        "seed=10/26-07-26-19_51_22",
        "seed=20/26-07-26-22_38_09",
    ],
    "NaiveFNO": [
        "seed=0/26-06-25-02:06:02",
        "seed=10/26-06-25-02:06:02",
        "seed=20/26-06-25-02:06:02",
    ],
    "SpatiotemporalFNO": [
        "seed=0/26-06-25-02:10:20",
        "seed=10/26-06-25-02:10:27",
        "seed=20/26-06-25-02:10:27",
    ],
}

## Query wandb API and obtain loss curves

In [ ]:
api = wandb.Api()
run_dpot = api.run("jhko725/hyperfluxfno/cgwa58nd")
run_hyperfluxfno = api.run("jhko725/hyperfluxfno/nni87so3")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
ax.plot(
    *run_dpot.history(keys=["train_loss"]).to_numpy().T,
    label="DPOT (ICML 2024)",
    color="royalblue",
)
ax.plot(
    *run_hyperfluxfno.history(keys=["train_loss"]).to_numpy().T,
    label="HyperFluxFNO (Ours)",
    color="orangered",
)
ax.set_yscale("log", base=10)
ax.set_ylabel("Train loss (MSE)")
ax.set_xlabel("Training steps")
ax.legend()
fig

## Evaluation on the in-distribution test set

In [ ]:
dataset_test = (
    xr.open_dataset(
        datadir
        / "datasets/cubic_no_source/data/test/cubic_no_source_large_test_seed=10.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel({"t": slice(0, 99)})
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for model_type, paths in model_paths.items():
    for p in tqdm(paths):
        model = load_model(checkpoint_dir / model_type / "OneStepLoss" / p)
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=6000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

In [ ]:
results_dict

with open("../../data/analysis/cubic_1d_indistribution.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [ ]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    stats = jax.tree.map(
        lambda list_: {
            "mean": jnp.mean(jnp.asarray(list_)),
            "std": jnp.std(jnp.asarray(list_)),
        },
        res_,
        is_leaf=lambda x: isinstance(x, list),
    )
    for k, v in stats.items():
        print(model_name, f"{k}: {v['mean']:.2e}" + "±" + f"{v['std']:.2e}")
    # print(
    #     model_name,
    #     jax.tree.map(
    #         lambda list_: {
    #             "mean": jnp.mean(jnp.asarray(list_)),
    #             "std": jnp.std(jnp.asarray(list_)),
    #         },
    #         res_,
    #         is_leaf=lambda x: isinstance(x, list),
    #     ),
    # )

In [ ]:
(jax.tree.transpose(jax.tree.structure(["*"] * 3), None, results_dict["DPOT"]),)

## Calculate results for the OOD dataset

In [ ]:
dataset_test = (
    xr.open_dataset(
        datadir / "datasets/cubic_no_source/data/test/sine_1d_test_step_seed=12.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel({"t": slice(0, 99)})
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for model_type, paths in model_paths.items():
    for p in tqdm(paths):
        model = load_model(checkpoint_dir / model_type / "OneStepLoss" / p)
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=6000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

with open("../../data/analysis/cubic_1d_ood_sine_step.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [ ]:
with open("../../data/analysis/cubic_1d_ood_sine_step.json", "r") as f:
    results = json.load(f)
results

In [ ]:
for model_name, res in results.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    print(
        model_name,
        jax.tree.map(
            lambda list_: {
                "mean": jnp.mean(jnp.asarray(list_)),
                "std": jnp.std(jnp.asarray(list_)),
            },
            res_,
            is_leaf=lambda x: isinstance(x, list),
        ),
    )

## Compare model performance for different training methods

In [ ]:
dataset_train = xr.open_dataset(
    datadir / "cubic_no_source_large_train.hdf5", engine="h5netcdf", chunks={}
)
dataset_train_extrap = dataset_train.isel({"t": slice(61, None)})

batch_extrap = dataset_train_extrap["values"].isel({"ic": 10}).compute().to_numpy()

dt = float(dataset_train_extrap["t"][1] - dataset_train_extrap["t"][0])
dx = float(dataset_train_extrap["x"][1] - dataset_train_extrap["x"][0])

### DPOT

In [ ]:
u_pred_dict = dict()

for method, ckpt_name in tqdm(CHECKPOINT_DICT["DPOT"].items()):
    model = load_model(checkpoint_dir / "DPOT" / method / ckpt_name)
    model = eqx.nn.inference_mode(model, True)

    u_pred_dict[method] = eqx.filter_vmap(
        lambda u: model.rollout(u, args=None, num_steps=20)
    )(batch_extrap[:, :20])[0]

sample_idx = 0
fig, axes = plt.subplots(
    1, 4, figsize=(7, 5), constrained_layout=True, sharex=True, sharey=True
)
axes[0].matshow(batch_extrap[sample_idx, 20:, 0].T)
axes[0].set_title("Data")
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    axes[i + 1].matshow(u_pred[sample_idx, :, 0].T)
    axes[i + 1].set_title(method)
fig.suptitle("DPOT")

In [ ]:
u_pred_dict = dict()

dt = float(dataset_train_extrap["t"][1] - dataset_train_extrap["t"][0])
dx = float(dataset_train_extrap["x"][1] - dataset_train_extrap["x"][0])

model_type = "HyperFluxFNO"
for method, ckpt_name in tqdm(CHECKPOINT_DICT[model_type].items()):
    model = load_model(checkpoint_dir / model_type / method / ckpt_name)
    model = eqx.nn.inference_mode(model, True)

    u_pred_dict[method] = eqx.filter_vmap(
        lambda u: model.rollout(u, args=(dt, dx), num_steps=20)
    )(batch_extrap[:, :20])[0]

sample_idx = 0
fig, axes = plt.subplots(
    1, 4, figsize=(7, 5), constrained_layout=True, sharex=True, sharey=True
)
axes[0].matshow(batch_extrap[sample_idx, 20:, 0].T)
axes[0].set_title("Data")
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    axes[i + 1].matshow(u_pred[sample_idx, :, 0].T)
    axes[i + 1].set_title(method)
fig.suptitle(model_type)

In [ ]:
def relative_rmse(
    u_pred: Float[Array, "channels ..."], u_data: Float[Array, "channels ..."]
):
    return jnp.sqrt(jnp.mean((u_pred - u_data) ** 2)) / jnp.sqrt(jnp.mean(u_data**2))


sample_idx = 0
fig, ax = plt.subplots(1, 1, figsize=(5, 3), constrained_layout=True)
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    rel_rmse = jnp.mean(
        jax.vmap(jax.vmap(relative_rmse))(batch_extrap[:, 20:], u_pred),
        axis=0,
    )
    ax.plot(jnp.arange(len(rel_rmse)) * dt, rel_rmse, label=method)
ax.set_ylabel("Average relative RMSE")
ax.set_xlabel("Rollout time")
ax.legend()
ax.set_title(model_type)

In [ ]:
u_pred_dict = dict()

dt = float(dataset_train_extrap["t"][1] - dataset_train_extrap["t"][0])
dx = float(dataset_train_extrap["x"][1] - dataset_train_extrap["x"][0])

model_type = "HyperFluxFNOLocal"
for method, ckpt_name in tqdm(CHECKPOINT_DICT[model_type].items()):
    model = load_model(checkpoint_dir / model_type / method / ckpt_name)
    model = eqx.nn.inference_mode(model, True)

    u_pred_dict[method] = eqx.filter_vmap(
        lambda u: model.rollout(u, args=(dt, dx), num_steps=20)
    )(batch_extrap[:, :20])[0]

sample_idx = 0
fig, axes = plt.subplots(
    1, 4, figsize=(7, 5), constrained_layout=True, sharex=True, sharey=True
)
axes[0].matshow(batch_extrap[sample_idx, 20:, 0].T)
axes[0].set_title("Data")
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    axes[i + 1].matshow(u_pred[sample_idx, :, 0].T)
    axes[i + 1].set_title(method)
fig.suptitle(model_type)

In [ ]:
sample_idx = 0
fig, ax = plt.subplots(1, 1, figsize=(5, 3), constrained_layout=True)
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    rel_rmse = jnp.mean(
        jax.vmap(jax.vmap(relative_rmse))(batch_extrap[:, 20:], u_pred),
        axis=0,
    )
    ax.plot(jnp.arange(len(rel_rmse)) * dt, rel_rmse, label=method)
ax.set_ylabel("Average relative RMSE")
ax.set_xlabel("Rollout time")
ax.legend()
ax.set_title(model_type)

## Compare best results

In [ ]:
u_pred_dict = dict()

# model_method_list = [
#     ("DPOT", "DenoisingOneStepLoss"),
#     ("HyperFluxFNO", "PushforwardOneStepLoss"),
#     ("HyperFluxFNOLocal", "PushforwardOneStepLoss"),
# ]
model_paths = {
    "DPOT": "OneStepLoss/seed=0/26-04-10-14:49:05",
    "HyperFluxFNOLocal": "OneStepLoss/seed=0/26-04-16-00:31:47",
    "DISCO": "OneStepLoss/seed=0/26-04-10-21:26:22",
}
for model_type, path in tqdm(model_paths.items()):
    model = load_model(checkpoint_dir / model_type / path)
    model = eqx.nn.inference_mode(model, True)

    u_pred_dict[model_type] = eqx.filter_vmap(
        lambda u: model.rollout(u, args=(dt, dx), num_steps=20)
    )(batch_extrap[:, :20])[0]

In [ ]:
sample_idx = 100
fig, axes = plt.subplots(
    1, 4, figsize=(7, 5), constrained_layout=True, sharex=True, sharey=True
)
axes[0].matshow(batch_extrap[sample_idx, 20:, 0].T)
axes[0].set_title("Data")
for i, (model_type, u_pred) in enumerate(u_pred_dict.items()):
    axes[i + 1].matshow(u_pred[sample_idx, :, 0].T)
    axes[i + 1].set_title(model_type)

In [ ]:
plt.rcParams["animation.html"] = "jshtml"
fig, axes = plt.subplots(2, 1, figsize=(7, 4), sharex=True, constrained_layout=True)

COLOR_DICT = {
    "DPOT": "royalblue",
    "HyperFluxFNOLocal": "orangered",
    "DISCO": "magenta",
}
LABEL_DICT = {
    "DPOT": "DPOT (ICML 2024)",
    "HyperFluxFNOLocal": "HyperFluxNO (Ours)",
    "DISCO": "DISCO (ICML 2025)",
}


def make_frame(timestep: int):
    if timestep == 0:
        plot_kwargs = {"animated": False}
        axes[-1].set_xlabel("$x$")
        axes[0].set_ylabel("$u$")
        axes[1].set_ylabel("$u_{pred}-u_{data}$")
    else:
        plot_kwargs = {"animated": True}

    plot1 = axes[0].plot(
        batch_extrap[sample_idx, 20 + timestep, 0].T,
        label="Data",
        linewidth=2.0,
        color="black",
        alpha=0.9,
        **plot_kwargs,
    )[0]
    plots = []
    for model_type, u_pred in u_pred_dict.items():
        p0 = axes[0].plot(
            u_pred[sample_idx, timestep, 0].T,
            label=LABEL_DICT[model_type],
            alpha=0.8,
            linewidth=1.5,
            color=COLOR_DICT[model_type],
            **plot_kwargs,
        )[0]
        p1 = axes[1].plot(
            u_pred[sample_idx, timestep, 0].T
            - batch_extrap[sample_idx, 20 + timestep, 0].T,
            label=LABEL_DICT[model_type],
            alpha=0.8,
            linewidth=1.5,
            color=COLOR_DICT[model_type],
            **plot_kwargs,
        )[0]
        plots.append(p0)
        plots.append(p1)

    text_y = (
        jnp.max(batch_extrap[sample_idx, :, 0]) * 0.8
        + jnp.min(batch_extrap[sample_idx, :, 0]) * 0.2
    )
    text0 = axes[0].text(70, text_y, f"$t=0.4+{dt * timestep:.3f}$", fontsize="large")
    if timestep == 0:
        fig.legend(*axes[0].get_legend_handles_labels(), loc="outside center right")
    axes[0].set_title(f"Extrapolation for (pde, ic)={(sample_idx, 0)}")
    return (plot1, *plots, text0)


frames = [make_frame(i) for i in range(20)]


ani = ArtistAnimation(fig, frames, interval=50, blit=True)
# ani.save(f"../../figures/cubic_extrapolation_{sample_idx}_0.gif")
ani

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3), constrained_layout=True)
for i, (model_type, u_pred) in enumerate(u_pred_dict.items()):
    rel_L2 = jnp.mean(
        jax.vmap(jax.vmap(relative_L2_error))(batch_extrap[:, 20:], u_pred),
        axis=0,
    )
    axes[0].plot(jnp.arange(len(rel_L2)) * dt, rel_L2, label=model_type)

    rel_L_infty = jnp.mean(
        jax.vmap(jax.vmap(relative_L_infty_error))(batch_extrap[:, 20:], u_pred),
        axis=0,
    )
    axes[1].plot(jnp.arange(len(rel_L_infty)) * dt, rel_L_infty, label=model_type)
axes[0].set_ylabel(r"Relative $L_2$ Error")
axes[1].set_ylabel(r"Relative $L_\infty$ Error")
fig.supxlabel("Rollout time", fontsize="medium")
axes[0].legend()

## Evaluation on the test set

In [ ]:
dataset_test = xr.open_dataset(
    datadir
    / "datasets/cubic_no_source/data/test/cubic_no_source_large_test_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
).isel(t=slice(None, None, 10))

# batch_test = dataset_test["values"].isel({"ic": 10}).compute().to_numpy()
# num_steps = 50

In [ ]:
dataset_test

In [ ]:
batch_test = dataset_test["values"].isel({"ic": 10}).compute().to_numpy()
batch_test.shape

In [ ]:
num_steps = 10
u_pred_dict = dict()

for method, ckpt_name in tqdm(CHECKPOINT_DICT["DPOT"].items()):
    model = load_model(checkpoint_dir / "DPOT" / method / ckpt_name)
    model = eqx.nn.inference_mode(model, True)

    u_pred_dict[method] = eqx.filter_vmap(
        lambda u: model.rollout(u, args=None, num_steps=num_steps)
    )(batch_test[:, :20])[0]

sample_idx = 0
fig, axes = plt.subplots(
    1, 4, figsize=(7, 5), constrained_layout=True, sharex=True, sharey=True
)
axes[0].matshow(batch_test[sample_idx, 20 : 20 + num_steps, 0].T)
axes[0].set_title("Data")
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    axes[i + 1].matshow(u_pred[sample_idx, :, 0].T)
    axes[i + 1].set_title(method)
fig.suptitle("DPOT")

In [ ]:
u_pred_dict = dict()

dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

model_type = "HyperFluxFNO"
for method, ckpt_name in tqdm(CHECKPOINT_DICT[model_type].items()):
    model = load_model(checkpoint_dir / model_type / method / ckpt_name)
    model = eqx.nn.inference_mode(model, True)

    u_pred_dict[method] = eqx.filter_vmap(
        lambda u: model.rollout(u, args=(dt, dx), num_steps=num_steps)
    )(batch_test[:, :20])[0]

sample_idx = 0
fig, axes = plt.subplots(
    1, 4, figsize=(7, 5), constrained_layout=True, sharex=True, sharey=True
)
axes[0].matshow(batch_test[sample_idx, 20 : 20 + num_steps, 0].T)
axes[0].set_title("Data")
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    axes[i + 1].matshow(u_pred[sample_idx, :, 0].T)
    axes[i + 1].set_title(method)
fig.suptitle(model_type)

In [ ]:
def relative_rmse(
    u_pred: Float[Array, "channels ..."], u_data: Float[Array, "channels ..."]
):
    return jnp.sqrt(jnp.mean((u_pred - u_data) ** 2)) / jnp.sqrt(jnp.mean(u_data**2))


sample_idx = 0
fig, ax = plt.subplots(1, 1, figsize=(5, 3), constrained_layout=True)
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    rel_rmse = jnp.mean(
        jax.vmap(jax.vmap(relative_rmse))(batch_test[:, 20 : 20 + num_steps], u_pred),
        axis=0,
    )
    ax.plot(jnp.arange(len(rel_rmse)) * dt, rel_rmse, label=method)
ax.set_ylabel("Average relative RMSE")
ax.set_xlabel("Rollout time")
ax.legend()
ax.set_title(model_type)

In [ ]:
u_pred_dict = dict()

dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

model_type = "HyperFluxFNOLocal"
for method, ckpt_name in tqdm(CHECKPOINT_DICT[model_type].items()):
    model = load_model(checkpoint_dir / model_type / method / ckpt_name)
    model = eqx.nn.inference_mode(model, True)

    u_pred_dict[method] = eqx.filter_vmap(
        lambda u: model.rollout(u, args=(dt, dx), num_steps=num_steps)
    )(batch_test[:, :20])[0]

sample_idx = 0
fig, axes = plt.subplots(
    1, 4, figsize=(7, 5), constrained_layout=True, sharex=True, sharey=True
)
axes[0].matshow(batch_test[sample_idx, 20 : 20 + num_steps, 0].T)
axes[0].set_title("Data")
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    axes[i + 1].matshow(u_pred[sample_idx, :, 0].T)
    axes[i + 1].set_title(method)
fig.suptitle(model_type)

In [ ]:
def relative_rmse(
    u_pred: Float[Array, "channels ..."], u_data: Float[Array, "channels ..."]
):
    return jnp.sqrt(jnp.mean((u_pred - u_data) ** 2)) / jnp.sqrt(jnp.mean(u_data**2))


sample_idx = 0
fig, ax = plt.subplots(1, 1, figsize=(5, 3), constrained_layout=True)
for i, (method, u_pred) in enumerate(u_pred_dict.items()):
    rel_rmse = jnp.mean(
        jax.vmap(jax.vmap(relative_rmse))(batch_test[:, 20 : 20 + num_steps], u_pred),
        axis=0,
    )
    ax.plot(jnp.arange(len(rel_rmse)) * dt, rel_rmse, label=method)
ax.set_ylabel("Average relative RMSE")
ax.set_xlabel("Rollout time")
ax.legend()
ax.set_title(model_type)

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

for model_type, path in tqdm(model_paths.items()):
    model = load_model(checkpoint_dir / model_type / path)
    model = eqx.nn.inference_mode(model, True)

    u_pred_dict[model_type] = eqx.filter_vmap(
        lambda u: model.rollout(u, args=(dt, dx), num_steps=20)
    )(batch_test[:, :20])[0]

In [ ]:
sample_idx = 20
fig, axes = plt.subplots(
    1, 4, figsize=(7, 5), constrained_layout=True, sharex=True, sharey=True
)
axes[0].matshow(batch_test[sample_idx, 20:40, 0].T)
axes[0].set_title("Data")
for i, (model_type, u_pred) in enumerate(u_pred_dict.items()):
    axes[i + 1].matshow(u_pred[sample_idx, :, 0].T)
    axes[i + 1].set_title(model_type)

In [ ]:
plt.rcParams["animation.html"] = "jshtml"
fig, axes = plt.subplots(2, 1, figsize=(7, 4), sharex=True, constrained_layout=True)

COLOR_DICT = {
    "DPOT": "royalblue",
    "HyperFluxFNOLocal": "orangered",
    "DISCO": "magenta",
}
LABEL_DICT = {
    "DPOT": "DPOT (ICML 2024)",
    "HyperFluxFNOLocal": "HyperFluxNO (Ours)",
    "DISCO": "DISCO (ICML 2025)",
}


def make_frame(timestep: int):
    if timestep == 0:
        plot_kwargs = {"animated": False}
        axes[-1].set_xlabel("$x$")
        axes[0].set_ylabel("$u$")
        axes[1].set_ylabel("$u_{pred}-u_{data}$")
    else:
        plot_kwargs = {"animated": True}

    plot1 = axes[0].plot(
        batch_test[sample_idx, 20 + timestep, 0].T,
        label="Data",
        linewidth=2.0,
        color="black",
        alpha=0.9,
        **plot_kwargs,
    )[0]
    plots = []
    for model_type, u_pred in u_pred_dict.items():
        p0 = axes[0].plot(
            u_pred[sample_idx, timestep, 0].T,
            label=LABEL_DICT[model_type],
            alpha=0.8,
            linewidth=1.5,
            color=COLOR_DICT[model_type],
            **plot_kwargs,
        )[0]
        p1 = axes[1].plot(
            u_pred[sample_idx, timestep, 0].T
            - batch_test[sample_idx, 20 + timestep, 0].T,
            label=LABEL_DICT[model_type],
            alpha=0.8,
            linewidth=1.5,
            color=COLOR_DICT[model_type],
            **plot_kwargs,
        )[0]
        plots.append(p0)
        plots.append(p1)

    text_y = (
        jnp.max(batch_test[sample_idx, :, 0]) * 0.8
        + jnp.min(batch_test[sample_idx, :, 0]) * 0.2
    )
    text0 = axes[0].text(70, text_y, f"$t=0.4+{dt * timestep:.3f}$", fontsize="large")
    if timestep == 0:
        fig.legend(*axes[0].get_legend_handles_labels(), loc="outside center right")
    axes[0].set_title(f"Extrapolation for (pde, ic)={(sample_idx, 0)}")
    return (plot1, *plots, text0)


frames = [make_frame(i) for i in range(20)]


ani = ArtistAnimation(fig, frames, interval=50, blit=True)
ani.save(f"../../figures/cubic_test_{sample_idx}_0.gif")
ani

In [ ]:
model_paths = {
    "DPOT": "OneStepLoss/seed=0/26-04-10-14:49:05",
    "HyperFluxFNOLocal": "OneStepLoss/seed=0/26-04-16-00:31:47",
    "DISCO": "OneStepLoss/seed=0/26-04-10-21:26:22",
}

In [ ]:
import numpy as np
from einops import rearrange


dataset_test = (
    xr.open_dataset(
        datadir
        / "datasets/cubic_no_source/data/test/cubic_no_source_large_test_seed=10.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel(t=slice(0, 80))
)

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 21, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results = {}
for model_type, path in tqdm(model_paths.items()):
    model = load_model(checkpoint_dir / model_type / path)
    model = eqx.nn.inference_mode(model, True)

    @eqx.filter_jit
    def _compute_metrics(_u):
        _u_pred = model(_u[:-1], (dt, dx))[0]
        return {
            "L2": relative_L2_error(_u_pred, _u[-1]),
            "L_infty": relative_L_infty_error(_u_pred, _u[-1]),
        }

    results[model_type] = jax.tree.map(
        jnp.mean, jax.lax.map(_compute_metrics, segments, batch_size=2000)
    )

In [ ]:
results

In [ ]:
dataset_test = (
    xr.open_dataset(
        datadir
        / "datasets/cubic_no_source/data/test/cubic_no_source_large_test_seed=10.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel(t=slice(80, None))
)

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 21, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results = {}
for model_type, path in tqdm(model_paths.items()):
    model = load_model(checkpoint_dir / model_type / path)
    model = eqx.nn.inference_mode(model, True)

    @eqx.filter_jit
    def _compute_metrics(_u):
        _u_pred = model(_u[:-1], (dt, dx))[0]
        return {
            "L2": relative_L2_error(_u_pred, _u[-1]),
            "L_infty": relative_L_infty_error(_u_pred, _u[-1]),
        }

    results[model_type] = jax.tree.map(
        jnp.mean, jax.lax.map(_compute_metrics, segments, batch_size=2000)
    )

In [ ]:
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3), constrained_layout=True)
for i, (model_type, u_pred) in enumerate(u_pred_dict.items()):
    rel_L2 = jnp.mean(
        jax.vmap(jax.vmap(relative_L2_error))(batch_test[:, 20:40], u_pred),
        axis=0,
    )
    axes[0].plot(jnp.arange(len(rel_L2)) * dt, rel_L2, label=model_type)

    rel_L_infty = jnp.mean(
        jax.vmap(jax.vmap(relative_L_infty_error))(batch_test[:, 20:40], u_pred),
        axis=0,
    )
    axes[1].plot(jnp.arange(len(rel_L_infty)) * dt, rel_L_infty, label=model_type)
axes[0].set_ylabel(r"Relative $L_2$ Error")
axes[1].set_ylabel(r"Relative $L_\infty$ Error")
fig.supxlabel("Rollout time", fontsize="medium")
axes[0].legend()

In [ ]:
dataset_test = (
    xr.open_dataset(
        datadir / "datasets/cubic_no_source/data/test/sine_1d_test_grf_seed=12.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel(t=slice(None, 80))
)

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 21, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results = {}
for model_type, path in tqdm(model_paths.items()):
    model = load_model(checkpoint_dir / model_type / path)
    model = eqx.nn.inference_mode(model, True)

    @eqx.filter_jit
    def _compute_metrics(_u):
        _u_pred = model(_u[:-1], (dt, dx))[0]
        return {
            "L2": relative_L2_error(_u_pred, _u[-1]),
            "L_infty": relative_L_infty_error(_u_pred, _u[-1]),
        }

    results[model_type] = jax.tree.map(
        jnp.mean, jax.lax.map(_compute_metrics, segments, batch_size=2000)
    )

In [ ]:
results

In [ ]:
dataset_test

In [ ]:
dataset_test = xr.open_dataset(
    datadir / "datasets/cubic_no_source/data/test/sine_1d_test_step_seed=12.hdf5",
    engine="h5netcdf",
    chunks={},
).isel(t=slice(None, None, 10))

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
values.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results = {}
for model_type, path in tqdm(model_paths.items()):
    model = load_model(checkpoint_dir / model_type / path)
    model = eqx.nn.inference_mode(model, True)

    results[model_type] = jax.lax.map(
        eqx.filter_jit(lambda u: model.rollout(u, args=(dt, dx), num_steps=81)[0]),
        values[:, :20],
        batch_size=1000,
    )

In [ ]:
metrics_rollout = {}
for model_type, u_pred in results.items():
    l2 = jnp.mean(
        eqx.filter_vmap(eqx.filter_vmap(relative_L2_error))(u_pred, values[:, 20:]),
        axis=0,
    )
    l_inf = jnp.mean(
        eqx.filter_vmap(eqx.filter_vmap(relative_L_infty_error))(
            u_pred, values[:, 20:]
        ),
        axis=0,
    )
    metrics_rollout[model_type] = {"l2": l2, "l_inf": l_inf}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3), constrained_layout=True)
for model_type, met in metrics_rollout.items():
    axes[0].plot(jnp.arange(len(met["l2"])) * dt, met["l2"], label=model_type)

    axes[1].plot(jnp.arange(len(met["l2"])) * dt, met["l_inf"], label=model_type)
axes[0].set_ylabel(r"Relative $L_2$ Error")
axes[1].set_ylabel(r"Relative $L_\infty$ Error")
fig.supxlabel("Rollout time", fontsize="medium")
axes[0].legend()

In [ ]:
dt

## Evaluate l2, l\infty losses

In [ ]:
import numpy as np
from einops import rearrange


dataset_test = xr.open_dataset(
    datadir / "cubic_no_source_large_test_seed=10.hdf5", engine="h5netcdf", chunks={}
).isel(t=slice(None, None, 10))


sample = dataset_test["values"].isel({"pde": 0}).compute()
inputs = np.lib.stride_tricks.sliding_window_view(sample, (21,), axis=1)
inputs.shape

In [ ]:
# model = load_model(
#     checkpoint_dir / "DPOT" / "OneStepLoss" / CHECKPOINT_DICT["DPOT"]["OneStepLoss"]
# )
model = load_model(
    checkpoint_dir
    / "HyperFluxFNO"
    / "OneStepLoss"
    / CHECKPOINT_DICT["HyperFluxFNO"]["OneStepLoss"]
)
model = eqx.nn.inference_mode(model, True)

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])


@eqx.filter_jit
def compute_metrics(model, data):
    u_pred = eqx.filter_vmap(model, in_axes=(0, None))(data[:, :-1], (dt, dx))[0]
    u_target = data[:, -1]
    l2 = jnp.mean(jnp.sqrt(jnp.mean((u_pred - u_target) ** 2, axis=-1)))
    l_infty = jnp.mean(jnp.max(jnp.abs(u_pred - u_target), axis=-1))
    return {"l2": l2, "l_infty": l_infty}

In [ ]:
metrics = []
for i in tqdm(range(dataset_test["values"].shape[0])):
    sample = dataset_test["values"].isel({"pde": i}).compute()
    inputs = np.lib.stride_tricks.sliding_window_view(sample, (21,), axis=1)
    inputs_batched = rearrange(inputs, "ic t_start c x t -> (ic t_start) t c x")
    metrics.append(compute_metrics(model, inputs_batched))

In [ ]:
metrics_mean = jax.tree.map(
    lambda x: {"mean": np.mean(x), "std": np.std(x)},
    jax.tree.transpose(jax.tree.structure(["*"] * len(metrics)), None, metrics),
    is_leaf=lambda x: isinstance(x, list),
)

In [ ]:
metrics_mean